#  Análisis de Componentes Principales (PCA)

Exploraremos lo que es, quizás, uno de los algoritmos no supervisados más utilizados: el análisis de componentes principales (PCA).

PCA es, en esencia, un algoritmo de reducción de dimensionalidad, pero también puede ser útil como herramienta de visualización, filtrado de ruido, extracción e ingeniería de características, y mucho más.
Después de una breve discusión conceptual del algoritmo PCA, exploraremos un par de ejemplos de estas aplicaciones adicionales.

Comenzamos con las importaciones estándar:

In [ ]:
#
import numpy as np
import matplotlib.pyplot as plt


## Introducción al Análisis de Componentes Principales

El análisis de componentes principales es un método rápido y flexible para reducir la dimensionalidad.

Su comportamiento es más fácil de visualizar observando un conjunto de datos de dos dimensiones.
Consideremos estos 200 puntos (ver la siguiente figura):

In [ ]:
#
rng = np.random.RandomState(1)
X = np.dot(rng.rand(2, 2), rng.randn(2, 200)).T

X.shape

In [ ]:
#
plt.scatter(X[:, 0], X[:, 1])
plt.axis('equal')
plt.xlabel('Disfrute')
plt.ylabel('Habilidad')
plt.title('Simulación de la relación entre disfrute y habilidades (realizando tu trabajo)')
plt.show()

En el análisis de componentes principales, esta relación se cuantifica encontrando una lista de los *ejes principales* de los datos, y usando esos ejes para describir el conjunto de datos.
Usando el estimador `PCA` de Scikit-Learn, podemos calcular esto de la siguiente forma:

In [ ]:
#
from sklearn.decomposition import PCA
pca = PCA(n_components=2)
pca.fit(X)

El ajuste (`fit`) aprende algunas cantidades a partir de los datos; las más importantes son los componentes y la varianza explicada:

In [ ]:
pca.components_

In [ ]:
#
#pca.explained_variance_
pca.explained_variance_ratio_

Para ver qué significan estos números, visualicémoslos como vectores sobre los datos de entrada, usando los componentes para definir la dirección del vector y la varianza explicada para definir el largo al cuadrado del vector (ver la siguiente figura):

In [ ]:
#
def draw_vector(v0, v1, ax=None):
    ax = ax or plt.gca()
    arrowprops=dict(arrowstyle='->', linewidth=2,
                    shrinkA=0, shrinkB=0)
    ax.annotate('', v1, v0, arrowprops=arrowprops)


In [ ]:
#
pca.explained_variance_

In [ ]:
#
pca.components_

In [ ]:
# graficar los datos
plt.scatter(X[:, 0], X[:, 1], alpha=0.2)
for length, vector in zip(pca.explained_variance_, pca.components_):
    v = vector * 3 * np.sqrt(length)
    draw_vector(pca.mean_, pca.mean_ + v)
plt.axis('equal');

Esta transformación de los ejes originales de los datos a los ejes principales es una *transformación afín*, lo que significa que está compuesta por una traslación, una rotación y un escalamiento uniforme.

Aunque este algoritmo para encontrar componentes principales pueda parecer solo una curiosidad matemática, resulta tener aplicaciones de gran alcance en el mundo del aprendizaje automático y la exploración de datos.

### PCA como reducción de dimensionalidad

Usar PCA para reducir la dimensionalidad consiste en poner en cero uno o más de los componentes principales más pequeños, lo que da como resultado una proyección de menor dimensión de los datos que preserva la máxima varianza posible.

Aquí hay un ejemplo de cómo usar PCA como una transformación de reducción de dimensionalidad:

In [ ]:
#
pca = PCA(n_components=1)
pca.fit(X)
X_pca = pca.transform(X)
print("forma original:    ", X.shape)
print("forma transformada:", X_pca.shape)

Los datos transformados se han reducido a una sola dimensión.
Para entender el efecto de esta reducción de dimensionalidad, podemos aplicar la transformación inversa a estos datos reducidos y graficarlos junto con los datos originales (ver la siguiente figura):

In [ ]:
#
X_new = pca.inverse_transform(X_pca)
plt.scatter(X[:, 0], X[:, 1], alpha=0.2)
plt.scatter(X_new[:, 0], X_new[:, 1], alpha=0.8)
plt.axis('equal');

Los puntos claros son los datos originales, mientras que los puntos oscuros son la versión proyectada.
Esto deja claro qué significa una reducción de dimensionalidad con PCA: se elimina la información a lo largo del eje (o ejes) principal menos importante, dejando solo el o los componentes de los datos con mayor varianza.
La fracción de varianza que se elimina (proporcional a la dispersión de los puntos alrededor de la línea formada en la figura anterior) es, aproximadamente, una medida de cuánta "información" se descarta en esta reducción de dimensionalidad.

Este conjunto de datos de dimensión reducida es, en cierto sentido, "suficientemente bueno" para codificar las relaciones más importantes entre los puntos: a pesar de reducir el número de características de los datos en un 50%, las relaciones generales entre los puntos se preservan en su mayoría.

### PCA para visualización: dígitos escritos a mano

La utilidad de la reducción de dimensionalidad puede no ser del todo evidente en solo dos dimensiones, pero se vuelve clara al observar datos de alta dimensionalidad.
Para verlo, echemos un vistazo rápido a la aplicación de PCA al conjunto de datos de dígitos.

Comenzaremos cargando los datos:

In [ ]:
#
from sklearn.datasets import load_digits
digits = load_digits()
digits.data.shape

In [ ]:
#
fig, axes = plt.subplots(10, 10, figsize=(8, 8),
                         subplot_kw={'xticks':[], 'yticks':[]},
                         gridspec_kw=dict(hspace=0.1, wspace=0.1))
for i, ax in enumerate(axes.flat):
    ax.imshow(digits.images[i], cmap='binary', interpolation='nearest', clim=(0, 16))#

In [ ]:
#
print(digits.data[:5])

Recordemos que el conjunto de datos de dígitos consiste en imágenes de 8 × 8 píxeles, es decir, son de 64 dimensiones.
Para ganar algo de intuición sobre las relaciones entre estos puntos, podemos usar PCA para proyectarlos en un número de dimensiones más manejable, digamos dos:

In [ ]:
#
pca = PCA(2)  # proyectar de 64 a 2 dimensiones
projected = pca.fit_transform(digits.data)
print(digits.data.shape)
print(projected.shape)

In [ ]:
#
projected

In [ ]:
#
pca.explained_variance_ratio_

Ahora podemos graficar los dos primeros componentes principales de cada punto para aprender sobre los datos, como se ve en la siguiente figura:

In [ ]:
#
plt.scatter(projected[:, 0], projected[:, 1],
            c=digits.target, edgecolor='none', alpha=0.5,
            cmap=plt.cm.get_cmap('rainbow', 10))
plt.xlabel('componente 1')
plt.ylabel('componente 2')
plt.colorbar();

Recordemos qué significan estos componentes: los datos completos son una nube de puntos en 64 dimensiones, y estos puntos son la proyección de cada dato a lo largo de las direcciones con mayor varianza.
En esencia, hemos encontrado el estiramiento y la rotación óptimos en un espacio de 64 dimensiones que nos permiten ver la disposición de los datos en dos dimensiones, y lo hemos hecho de manera no supervisada, es decir, sin usar las etiquetas.

### Eligiendo el número de componentes

Una parte fundamental de usar PCA en la práctica es poder estimar cuántos componentes se necesitan para describir los datos.
Esto se puede determinar observando la *proporción de varianza explicada* acumulada en función del número de componentes (ver la siguiente figura):

In [ ]:
#
pca = PCA().fit(digits.data)
plt.plot(np.cumsum(pca.explained_variance_ratio_))
plt.xlabel('número de componentes')
plt.ylabel('varianza explicada acumulada');

Esta curva cuantifica qué proporción de la varianza total, en las 64 dimensiones, está contenida en los primeros $N$ componentes.
Por ejemplo, vemos que con los datos de dígitos los primeros 10 componentes contienen aproximadamente el 75% de la varianza, mientras que se necesitan alrededor de 50 componentes para describir cerca del 100% de la varianza.

Esto nos dice que nuestra proyección en 2 dimensiones pierde mucha información (medida por la varianza explicada) y que necesitaríamos cerca de 20 componentes para conservar el 90% de la varianza. Observar esta gráfica para un conjunto de datos de alta dimensionalidad puede ayudarte a entender el nivel de redundancia presente en sus características.

In [ ]:
import pandas as pd

# Aplicar PCA con 10 componentes
pca_10 = PCA(n_components=10)
projected_10 = pca_10.fit_transform(digits.data)

# Crear un DataFrame con los datos proyectados
df_pca_10 = pd.DataFrame(projected_10, columns=[f'PC{i+1}' for i in range(10)])

In [ ]:
print("Forma de los datos originales:", digits.data.shape)
print("Forma del DataFrame transformado:", df_pca_10.shape)
display(df_pca_10.head())

In [ ]:
# Guardar el DataFrame en un archivo de Excel
df_pca_10.to_excel('df_pca_10.xlsx', index=False)

print("DataFrame df_pca_10 guardado en df_pca_10.xlsx")

## Para pensar

1. Revisa la curva de varianza explicada acumulada para el dataset de dígitos (64 dimensiones). ¿Cuántas componentes se necesitan para capturar el 90% de la varianza total? ¿Qué ganas y qué pierdes al usar ese número de componentes en vez de las 64 dimensiones originales?

2. PCA elige las direcciones de mayor varianza **sin ver ninguna etiqueta** (es no supervisado). ¿Es posible que una dirección con poca varianza sea, sin embargo, muy útil para distinguir clases (por ejemplo, separar el dígito "1" del "7")? ¿Qué implica esto sobre usar PCA como paso previo a un clasificador?

3. **Ejercicio:** repite la visualización 2D de los dígitos usando 3 componentes en vez de 2 (puedes graficar pares de componentes, p. ej. componente 1 vs. 3). ¿Se separan mejor o peor los dígitos que en la proyección a 2 componentes?
